# Clustering No Supervisado — Artículos MTSK
## Pipeline: SBERT → UMAP (cosine, 5D) → HDBSCAN
### Enfoque exploratorio puro: el algoritmo descubre la estructura temática natural

> **Justificación metodológica:** Se usa HDBSCAN en lugar de KMeans porque:
> 1. HDBSCAN respeta la geometría no lineal producida por UMAP (no asume clusters esféricos)
> 2. Descubre el número de grupos por sí solo, sin imponer K a priori
> 3. Identifica artículos atípicos (outliers) que no pertenecen a ningún grupo
> 4. Es el algoritmo recomendado por los propios autores de UMAP (McInnes et al., 2018)
>
> La correspondencia con la clasificación supervisada (T1-T5) se analiza **a posteriori**
> mediante matriz de contingencia, ARI y NMI — sin forzar el número de grupos.


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELDA 1 — Instalación de librerías
# Ejecutar solo una vez — Colab se reinicia automáticamente
# ─────────────────────────────────────────────────────────────
!pip install -q sentence-transformers==2.7.0
!pip install -q datasets==2.14.0
!pip install -q umap-learn
!pip install -q hdbscan

import IPython
IPython.Application.instance().kernel.do_shutdown(True)


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELDA 2 — Imports
# Pipeline limpio: sin PCA, sin KMeans como método principal
# ─────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import normalize, LabelEncoder
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.metrics.pairwise import cosine_similarity
from scipy.cluster.hierarchy import dendrogram, linkage
import umap
import hdbscan

print("✅ Librerías cargadas")
print("   Pipeline: SBERT → normalize → UMAP(cosine) → HDBSCAN")


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELDA 3 — Montar Drive y cargar dataset
# ─────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

RUTA_DATOS = '/content/drive/MyDrive/TG Maestria/No supervisado/DATACEROMTSK_AUMENTADO_SE.xlsx'

df = pd.read_excel(RUTA_DATOS)
print("Shape:", df.shape)
print("Columnas:", df.columns.tolist())
df.head(3)


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELDA 4 — Preprocesamiento de texto
# título × 2 y keywords × 2 para dar mayor peso semántico
# ─────────────────────────────────────────────────────────────
def limpiar_texto(texto):
    if pd.isna(texto):
        return ""
    texto = str(texto)
    texto = re.sub(r'\n', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

df['titulo_c']   = df['Título'].apply(limpiar_texto)
df['resumen_c']  = df['Resumen (texto literal)'].apply(limpiar_texto)
df['keywords_c'] = df['Palabras clave (texto literal)'].apply(limpiar_texto)

df['texto_completo'] = (
    df['titulo_c']   + '. ' + df['titulo_c']   + '. ' +
    df['resumen_c']  + '. ' +
    df['keywords_c'] + '. ' + df['keywords_c']
)

print(f"Total artículos: {len(df)}")
print(f"Textos vacíos:   {df['texto_completo'].str.strip().eq('').sum()}")
print("\nEjemplo:")
print(df['texto_completo'].iloc[0][:300])


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELDA 5 — Embeddings multilingüe con SBERT
# paraphrase-multilingual-MiniLM-L12-v2:
#   • Español, inglés y portugués en el mismo espacio vectorial
#   • 384 dimensiones por artículo
#   • Similitud semántica codificada en el ángulo (coseno)
# ─────────────────────────────────────────────────────────────
print("Cargando modelo SBERT multilingüe...")
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

print("Generando embeddings (1-2 minutos)...")
textos     = df['texto_completo'].tolist()
embeddings = model.encode(textos, show_progress_bar=True, batch_size=32)

# Normalización L2: norma de cada vector = 1
# Hace que distancia euclidiana ≈ complemento de similitud coseno
# Necesario porque HDBSCAN usará métrica euclidiana sobre X
X = normalize(embeddings)

print(f"\n✅ Embeddings generados: {embeddings.shape}")
print(f"   Normas (deben ser ≈ 1.0): "
      f"{np.linalg.norm(X, axis=1).min():.4f} – {np.linalg.norm(X, axis=1).max():.4f}")


## Reducción de dimensionalidad — UMAP directo (sin PCA)

```
SBERT(384D) → normalize → UMAP(5D, metric='cosine') → HDBSCAN
                           UMAP(2D, metric='cosine') → Visualización
```

- **Sin PCA:** se elimina la etapa lineal intermedia que distorsionaba el manifold semántico
- **`metric='cosine'`:** captura directamente la similitud angular de los embeddings SBERT
- **5D para clustering, 2D para visualización:** entrenados por separado desde los embeddings originales


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELDA 6 — UMAP solo para visualización
# ─────────────────────────────────────────────────────────────

# UMAP 2D — exclusivamente para visualización
# No se usa para clusterizar — solo para ver los grupos en 2D
print("Calculando UMAP 5D para clustering...")
reducer_5d = umap.UMAP(
    n_components=5,
    n_neighbors=15,
    min_dist=0.0,
    metric='cosine',
    random_state=42
)
X_5d = reducer_5d.fit_transform(X)

print("Calculando UMAP 2D para visualización...")
reducer_2d = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric='cosine',
    random_state=42
)
X_2d = reducer_2d.fit_transform(X)


print(f"\n✅ UMAP 2D calculado: {X_2d.shape}")
print(f"   Esta proyección es SOLO para visualización")
print(f"   El clustering se hará sobre X ({X.shape}) directamente")

## Clustering con HDBSCAN

**¿Por qué HDBSCAN y no KMeans?**

| | KMeans | HDBSCAN |
|---|---|---|
| Forma de clusters | Solo esférica | Cualquier forma |
| Número de clusters | Debes definirlo tú | Lo descubre solo |
| Outliers | No detecta | Los marca como -1 |
| Consistencia con UMAP | Baja | Alta |

**Hiperparámetro clave — `min_cluster_size`:**
Define cuántos artículos mínimo debe tener un grupo para ser considerado cluster.
Se ajusta según el tamaño del corpus. Con ~293 artículos, valores entre 8 y 20 son razonables.


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELDA 7 — HDBSCAN: descubrimiento de grupos temáticos
# ─────────────────────────────────────────────────────────────
# Exploración con min_samples para reducir outliers
print(f"{'mcs':>5} {'min_samples':>12} {'N clusters':>11} {'N outliers':>11} {'% outliers':>11}")
print("─" * 55)

for min_s in [1, 2, 3, 5, 7]:
    cl = hdbscan.HDBSCAN(
        min_cluster_size=5,
        min_samples=min_s,
        metric='euclidean',
        cluster_selection_method='eom'
    )
    lbl = cl.fit_predict(X_5d)
    n_c = len(set(lbl)) - (1 if -1 in lbl else 0)
    n_o = (lbl == -1).sum()
    print(f"{'5':>5} {min_s:>12} {n_c:>11} {n_o:>11} {n_o/len(lbl)*100:>10.1f}%")


# ── Elegir min_cluster_size ──
# Ajusta este valor según la exploración anterior
# Busca un balance: pocos outliers y grupos interpretables
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=5,
    min_samples=5,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)
labels_hdb = clusterer.fit_predict(X_5d)   # ← X_5d, no X

n_clusters = len(set(labels_hdb)) - (1 if -1 in labels_hdb else 0)
n_outliers = (labels_hdb == -1).sum()

print(f"✅ HDBSCAN completado con min_cluster_size=5:")
print(f"   Grupos descubiertos: {n_clusters}")
print(f"   Outliers:            {n_outliers} artículos ({n_outliers/len(labels_hdb)*100:.1f}%)")
print(f"\nDistribución por grupo:")
for g in sorted(set(labels_hdb)):
    nombre = f"Grupo {g+1}" if g >= 0 else "Outliers"
    n = (labels_hdb == g).sum()
    barra = '█' * int(n / len(labels_hdb) * 40)
    print(f"  {nombre:>10}: {n:>3} artículos  {barra}")

df['grupo_hdb'] = labels_hdb


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELDA 8 — Métricas de calidad del clustering
# Calculadas sobre artículos no-outliers (labels != -1)
# y también sobre el espacio original de 384D con coseno
# ─────────────────────────────────────────────────────────────
mask_validos = labels_hdb != -1

if mask_validos.sum() < 10 or len(set(labels_hdb[mask_validos])) < 2:
    print("⚠️ Muy pocos puntos válidos para calcular métricas. Ajusta min_cluster_size.")
else:
    # Métricas en espacio UMAP 5D (donde se hizo el clustering)
    s_umap  = silhouette_score(X_5d[mask_validos], labels_hdb[mask_validos])
    db_umap = davies_bouldin_score(X_5d[mask_validos], labels_hdb[mask_validos])
    ch_umap = calinski_harabasz_score(X_5d[mask_validos], labels_hdb[mask_validos])

    # Métricas en espacio original 384D con coseno (más riguroso)
    s_orig  = silhouette_score(X[mask_validos], labels_hdb[mask_validos], metric='cosine')

    print("Métricas de calidad del clustering HDBSCAN:")
    print(f"\n  Espacio UMAP 5D:")
    print(f"    Silhouette Score:      {s_umap:.4f}  (↑ mejor, rango -1 a 1)")
    print(f"    Davies-Bouldin Index:  {db_umap:.4f}  (↓ mejor)")
    print(f"    Calinski-Harabasz:     {ch_umap:.2f}  (↑ mejor)")
    print(f"\n  Espacio original 384D (coseno) — más riguroso:")
    print(f"    Silhouette Score:      {s_orig:.4f}")
    print(f"\n  Nota: métricas calculadas sobre {mask_validos.sum()} artículos")
    print(f"  (excluyendo {n_outliers} outliers)")


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELDA 9 — Visualización UMAP 2D con grupos HDBSCAN
# ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

PALETA = ['#1D3557', '#E63946', '#2A9D8F', '#F4A261', '#6A0572',
          '#4CC9F0', '#F72585', '#06D6A0', '#FB8500', '#7B2D8B']
MARCADORES = ['o', 's', '^', 'D', 'P', 'h', '*', 'X', 'v', '<']

fig, ax = plt.subplots(figsize=(10, 8), facecolor='white')

grupos_unicos = sorted([g for g in set(labels_hdb) if g >= 0])

for i, g in enumerate(grupos_unicos):
    mask = labels_hdb == g
    ax.scatter(
        X_2d[mask, 0], X_2d[mask, 1],
        color=PALETA[i % len(PALETA)],
        marker=MARCADORES[i % len(MARCADORES)],
        s=75, alpha=0.85,
        edgecolors='white', linewidths=0.6,
        label=f'Grupo {g+1}  ({mask.sum()} artículos)',
        zorder=3
    )
    # Centroide
    cx, cy = X_2d[mask, 0].mean(), X_2d[mask, 1].mean()
    ax.scatter(cx, cy, color=PALETA[i % len(PALETA)], s=300,
               marker='*', edgecolors='black', linewidths=1.2, zorder=5)

# Outliers
if n_outliers > 0:
    mask_out = labels_hdb == -1
    ax.scatter(X_2d[mask_out, 0], X_2d[mask_out, 1],
               color='#CCCCCC', marker='x', s=50, alpha=0.6,
               linewidths=1.2, label=f'Outliers  ({n_outliers} artículos)',
               zorder=2)

ax.set_title(
    f'Agrupamiento HDBSCAN — {n_clusters} grupos descubiertos\n'
    f'Pipeline: SBERT(384D) → UMAP(5D, cosine) → HDBSCAN\n'
    f'Visualización sobre UMAP 2D independiente  |  ★ = centroide',
    fontsize=12, fontweight='bold', color='#1D3557', pad=12
)
ax.set_xlabel('UMAP Dimensión 1', fontsize=11)
ax.set_ylabel('UMAP Dimensión 2', fontsize=11)
ax.legend(fontsize=10, framealpha=0.95, loc='best')
ax.grid(True, alpha=0.2)
ax.set_facecolor('#FAFAFA')

plt.tight_layout()
plt.savefig('hdbscan_umap2d.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfica guardada: hdbscan_umap2d.png")


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELDA 10 — Interpretación semántica de cada grupo
# Artículos más cercanos al centroide (similitud coseno)
# en el espacio de embeddings originales
# ─────────────────────────────────────────────────────────────
print(f"{'='*70}")
print(f"INTERPRETACIÓN SEMÁNTICA — {n_clusters} grupos HDBSCAN")
print(f"{'='*70}\n")

for g in sorted([x for x in set(labels_hdb) if x >= 0]):
    mask   = labels_hdb == g
    subset = df[mask].reset_index(drop=True)

    # Centroide en espacio de embeddings originales
    centroide = embeddings[mask].mean(axis=0, keepdims=True)
    sims      = cosine_similarity(centroide, embeddings[mask])[0]
    top3      = sims.argsort()[-3:][::-1]

    print(f"{'─'*70}")
    print(f"GRUPO {g+1}  |  {mask.sum()} artículos  ({mask.sum()/len(df)*100:.1f}%)")
    print(f"{'─'*70}")
    print("Artículos más representativos (más cercanos al centroide semántico):")
    for i in top3:
        titulo = str(subset.loc[i, 'Título']).replace('\n', ' ').strip()
        print(f"  ⭐ {titulo[:85]}")
    print("\nOtros títulos del grupo:")
    otros = subset['Título'].tolist()
    for titulo in otros[:5]:
        titulo = str(titulo).replace('\n', ' ').strip()
        print(f"  •  {titulo[:85]}")
    print()

# Outliers
if n_outliers > 0:
    print(f"{'─'*70}")
    print(f"OUTLIERS  |  {n_outliers} artículos atípicos")
    print(f"{'─'*70}")
    print("Artículos que no pertenecen claramente a ningún grupo temático:")
    for titulo in df[labels_hdb == -1]['Título'].values:
        titulo = str(titulo).replace('\n', ' ').strip()
        print(f"  ✗  {titulo[:85]}")


## Comparación con clasificación supervisada

Se analiza la correspondencia **a posteriori** entre los grupos descubiertos por HDBSCAN
y la clasificación temática supervisada (T1–T5), sin haber forzado el número de grupos.

La matriz de contingencia muestra cuántos artículos de cada tema supervisado
cayeron en cada grupo no supervisado.


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELDA 11 — Matriz de contingencia: HDBSCAN vs supervisado
# ARI y NMI como métricas de acuerdo
# ─────────────────────────────────────────────────────────────
import matplotlib
matplotlib.rcParams.update({
    'font.size': 13,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 14,
    'axes.labelsize': 13,
})

RUTA_SUPERVISADO = '/content/drive/MyDrive/TG Maestria/DATACEROMTSK_AUMENTADO.xlsx'
RUTA_CLUSTERS    = '/content/drive/MyDrive/TG Maestria/No supervisado/clustering_articulos_MTSK.csv'

df_sup = pd.read_excel(RUTA_SUPERVISADO)
df_ns  = pd.read_csv(RUTA_CLUSTERS)

def limpiar(t):
    return re.sub(r'\s+', ' ', str(t).strip().lower())

df_sup['titulo_key'] = df_sup['Título'].apply(limpiar)
df_ns['titulo_key']  = df_ns['Título'].apply(limpiar)

col_tema = [c for c in df_sup.columns if 'lasif' in c or 'ematica' in c][0]
df_merged = df_ns.merge(df_sup[['titulo_key', col_tema]], on='titulo_key', how='left')
df_merged = df_merged.rename(columns={col_tema: 'Clasificacion_tematica'})
df_merged = df_merged.dropna(subset=['Clasificacion_tematica'])
df_merged = df_merged.drop_duplicates('titulo_key').reset_index(drop=True)

# Agregar etiqueta HDBSCAN
df['titulo_key'] = df['Título'].apply(limpiar)
df_merged = df_merged.merge(df[['titulo_key', 'grupo_hdb']], on='titulo_key', how='left')

print(f"Artículos cruzados exitosamente: {len(df_merged)}")
print(f"Temas supervisados encontrados:  {sorted(df_merged['Clasificacion_tematica'].unique())}")
print(f"Grupos HDBSCAN encontrados:      {sorted(df_merged['grupo_hdb'].dropna().unique())}")


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELDA 12 — Métricas ARI y NMI + Matriz de contingencia
# ─────────────────────────────────────────────────────────────

# Filtrar outliers para ARI/NMI (son ruido, no grupos)
df_sin_outliers = df_merged[df_merged['grupo_hdb'] >= 0].dropna(subset=['grupo_hdb'])

le      = LabelEncoder()
y_true  = le.fit_transform(df_sin_outliers['Clasificacion_tematica'])
y_pred  = df_sin_outliers['grupo_hdb'].astype(int).values

ari = adjusted_rand_score(y_true, y_pred)
nmi = normalized_mutual_info_score(y_true, y_pred)

print("Métricas de acuerdo entre HDBSCAN y clasificación supervisada:")
print(f"  ARI (Adjusted Rand Index):          {ari:.4f}")
print(f"  NMI (Normalized Mutual Information): {nmi:.4f}")
print()
print("Interpretación:")
print(f"  ARI = 1.0 → acuerdo perfecto")
print(f"  ARI = 0.0 → equivalente a asignación aleatoria")
print(f"  ARI < 0.0 → peor que el azar")
print(f"  NMI = 1.0 → información compartida total")
print(f"  NMI = 0.0 → grupos completamente independientes")

# ── Matriz de contingencia — conteos absolutos ──
# Etiquetas de filas: grupos HDBSCAN (incluyendo outliers si los hay)
df_all = df_merged.copy()
df_all['grupo_label'] = df_all['grupo_hdb'].apply(
    lambda x: f'G{int(x)+1}' if x >= 0 else 'Outlier'
)

mat = pd.crosstab(
    df_all['grupo_label'],
    df_all['Clasificacion_tematica']
)
mat_pct = (mat.div(mat.sum(axis=1), axis=0) * 100).round(1)

print(f"\nMatriz de contingencia (conteos absolutos):")
print(mat.to_string())
print(f"\nMatriz de contingencia (% por grupo):")
print(mat_pct.to_string())


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELDA 13 — Visualización: heatmaps de la matriz de contingencia
# Panel 1: conteos absolutos
# Panel 2: porcentaje por grupo (composición temática)
# ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, max(5, len(mat) * 1.2 + 2)),
                          facecolor='white')
fig.suptitle(
    'Correspondencia entre agrupamiento HDBSCAN y clasificación supervisada\n'
    f'ARI = {ari:.4f}   |   NMI = {nmi:.4f}',
    fontsize=14, fontweight='bold', color='#1D3557', y=1.02
)

grupos_filas = mat.index.tolist()
temas_cols   = mat.columns.tolist()

for ax, datos, cmap, titulo, fmt in [
    (axes[0], mat.values,     'Blues',  'Conteos absolutos\n(artículos por celda)', 'd'),
    (axes[1], mat_pct.values, 'YlOrRd', 'Composición temática por grupo\n(% dentro de cada grupo)', '.1f'),
]:
    im = ax.imshow(datos, cmap=cmap, aspect='auto',
                   vmin=0, vmax=datos.max())
    plt.colorbar(im, ax=ax, shrink=0.8,
                 label='Artículos' if fmt == 'd' else '% artículos')

    ax.set_xticks(range(len(temas_cols)))
    ax.set_xticklabels(temas_cols, fontsize=12, fontweight='bold')
    ax.set_yticks(range(len(grupos_filas)))
    ax.set_yticklabels(grupos_filas, fontsize=12, fontweight='bold')
    ax.set_xlabel('Clasificación Temática — Supervisado', fontsize=12, labelpad=10)
    ax.set_ylabel('Grupo — HDBSCAN (No Supervisado)', fontsize=12, labelpad=10)
    ax.set_title(titulo, fontsize=12, fontweight='bold', color='#1D3557', pad=10)

    umbral = datos.max() * 0.55
    for i in range(datos.shape[0]):
        for j in range(datos.shape[1]):
            val = datos[i, j]
            texto = str(int(val)) if fmt == 'd' else f'{val:.1f}%'
            color = 'white' if val > umbral else '#1D3557'
            ax.text(j, i, texto, ha='center', va='center',
                    fontsize=11, fontweight='bold', color=color)

plt.tight_layout()
plt.savefig('matriz_contingencia_hdbscan.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfica guardada: matriz_contingencia_hdbscan.png")


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELDA 14 — Árbol de probabilidades HDBSCAN
# Muestra la jerarquía de condensación de clusters
# ─────────────────────────────────────────────────────────────
clusterer.condensed_tree_.plot(
    select_clusters=True,
    selection_palette=PALETA[:n_clusters],
    label_clusters=True
)
plt.title('Árbol de condensación HDBSCAN\n'
          'Muestra cómo los grupos emergen a distintos niveles de densidad',
          fontsize=13, fontweight='bold', color='#1D3557')
plt.tight_layout()
plt.savefig('arbol_condensacion_hdbscan.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfica guardada: arbol_condensacion_hdbscan.png")


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELDA 15 — Exportar resultados completos
# ─────────────────────────────────────────────────────────────
df_export = df[['Título', 'Resumen (texto literal)', 'Palabras clave (texto literal)']].copy()
df_export['Grupo_HDBSCAN'] = labels_hdb + 1   # grupos desde 1; outliers = 0
df_export['Es_outlier']    = labels_hdb == -1
df_export['Grupo_label']   = df_export['Grupo_HDBSCAN'].apply(
    lambda x: f'Grupo {x}' if x > 0 else 'Outlier'
)
df_export = df_export.sort_values('Grupo_HDBSCAN').reset_index(drop=True)

RUTA_CSV = '/content/drive/MyDrive/TG Maestria/No supervisado/clustering_hdbscan_MTSK.csv'
df_export.to_csv(RUTA_CSV, index=False, encoding='utf-8-sig')

# Guardar arrays
np.save('embeddings_MTSK_multilingual.npy', embeddings)
np.save('X_umap_2d.npy', X_2d)
np.save('X_umap_5d.npy', X_5d)
np.save('labels_hdbscan.npy', labels_hdb)

print("✅ Exportación completa:")
print(f"   CSV: {RUTA_CSV}")
print(f"   Registros: {len(df_export)}")
print(f"\nDistribución final:")
print(df_export['Grupo_label'].value_counts().sort_index())
print(f"\nArrays .npy guardados para reutilización")


## Resumen metodológico final

| Etapa | Método | Entrada | Salida | Justificación |
|-------|--------|---------|--------|---------------|
| Embeddings | SBERT `paraphrase-multilingual-MiniLM-L12-v2` | Texto combinado | 384D | Representación semántica multilingüe |
| Normalización | L2 normalize | 384D | 384D (norma=1) | Distancia euclidiana ≈ complemento coseno |
| Reducción (clustering) | UMAP `metric='cosine'`, `min_dist=0.0` | 384D | 5D | No lineal puro, sin PCA |
| Reducción (visualiz.) | UMAP `metric='cosine'`, `min_dist=0.1` | 384D | 2D | Entrenado por separado |
| Agrupamiento | HDBSCAN `metric='euclidean'` | 5D | etiquetas | Respeta geometría UMAP, descubre K automáticamente |
| Evaluación interna | Silhouette, Davies-Bouldin, Calinski-Harabasz | 5D y 384D | escalar | Calidad del clustering sin supervisión |
| Evaluación externa | ARI, NMI + Matriz de contingencia | etiquetas | escalar + tabla | Correspondencia con clasificación supervisada |

**Decisiones clave:**
- Se eliminó PCA: transformación lineal incompatible con la geometría no lineal de UMAP
- Se reemplazó KMeans por HDBSCAN: respeta la topología del espacio UMAP
- K no se define a priori: HDBSCAN descubre la estructura temática natural del corpus
- La comparación con T1-T5 es a posteriori mediante matriz de contingencia
